# Lab 11: Government Data APIs

*INFO 4617/5617 · Week 11 · Chapter 11, [Government Data APIs](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html)*

This week every class is live coding: one lab in three parts, one a day. We use two public APIs, each with an idea that holds for APIs in general. On Monday and Wednesday, the **Census Bureau** and *what can I ask for?*: from a table on its website to the same number from its API, then many places at once, and whether to trust a number. On Friday, **FRED**, the Federal Reserve Bank of St. Louis's data service, and *let the server compute*. The chapter's third API, the EIA's, is for later, on your own (Step 14 says where).

Each code cell holds a comment that says what to write; cells marked `# Given` are already written, and we read and run them together. After each step, a **💬 Discuss** box holds questions to talk over; click it to open it. Short ⏱ microsprints ask you to change one thing, alone or in pairs.

**Before class:** open this notebook in the `webdata` environment (chapter 1), and have your email address ready for the User-Agent. Get two free keys ([Get Your Keys](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#get-your-keys)), and store each as an environment variable, as the chapter's ["Setting Environment Variables"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#setting-environment-variables) shows:
- `CENSUS_KEY`, from the [Census Bureau](https://api.census.gov/data/key_signup.html), for Monday and Wednesday; click the activation link in its email before you use it;
- `FRED_KEY`, from [FRED](https://fred.stlouisfed.org/docs/api/api_key.html), for Friday; FRED asks you to make a free account first, so allow a few minutes.

The simplest way is one line per key in a terminal (on Windows, the Anaconda Prompt), `conda env config vars set CENSUS_KEY="your-key" -n webdata`. Then **quit Jupyter** (restarting the kernel isn't enough), close the terminal, open a new one, run `conda activate webdata`, and start Jupyter from it. Treat a key like a password: never paste it into a cell or a code file.

On Wednesday and Friday, start at that day's setup step.

# Part 1 — Monday: the Census, and what can I ask for?

Every number on the Census Bureau's website also comes from its API. We start from one table on the website, find it in the API's own catalog, and ask the API for the website's number, by hand. Then we break the request on purpose, to see how the API says no.

## Step 0 — Set up, and check your key (6 minutes)

An API key identifies you, so treat it like a password: keep it out of every cell and file, and read it from an **environment variable**, a setting Jupyter receives when it starts ([Reading the Keys in Python](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#reading-the-keys-in-python)). Put your email in the first given cell and run it. If it says `MISSING`, quit Jupyter and start it from a new terminal ([When a Key Says `MISSING`](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#when-a-key-says-missing)), or run the next cell, which asks with [`getpass()`](https://docs.python.org/3/library/getpass.html).

In [ ]:
# Given: imports; your User-Agent (put your email in it); a check that CENSUS_KEY is set, not the key
import os
import time
import requests
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", None)   # show whole descriptions, not "..."
HEADERS = {"User-Agent": "WebDataScience/1.0 (your-email@colorado.edu)"}
# saved copies, for when a cell stops:
checkpoints = "https://raw.githubusercontent.com/cuinfoscience/INFO4617-Fall2026/main/labs/week-11/data/"
print("CENSUS_KEY:", "set" if os.environ.get("CENSUS_KEY") else "MISSING")

In [ ]:
# Given: only if the check said MISSING. A box opens: paste your key there and press Enter
# (nothing shows as you paste). If the check said set, this does nothing
from getpass import getpass
if not os.environ.get("CENSUS_KEY"):
    os.environ["CENSUS_KEY"] = getpass("CENSUS_KEY isn't set. Paste it here: ")

In [ ]:
# census_key: read "CENSUS_KEY" from os.environ with square brackets (never print it)
# Your code here

<details>
<summary>💬 Discuss</summary>

- In week 10 we told Wikipedia who we are with a User-Agent. What can a key do that a User-Agent can't?
- The Census Bureau takes the key as a parameter in the URL, so `response.url` holds it. Where could a printed URL end up, and could you take it back?
- Why doesn't a key set after Jupyter started reach the notebook, even after you restart the kernel?

</details>

## Step 1 — From the website to the catalog (11 minutes)

On [data.census.gov](https://data.census.gov/), table **B19013** gives Boulder County's median household income: \$103,994, ±\$2,268, in the American Community Survey's (ACS) five-year estimates for 2020–2024 ([figure 11.1](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#fig-census-table)). The API calls a table a **group**, and lists every group in a catalog, `groups.json`. This is our first request to the API, so print what came back before reading it. Then search the descriptions with [`str.contains()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.contains.html), as in ["From the Website's Table to the API's Group"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#from-the-websites-table-to-the-apis-group).

In [ ]:
# census_base: "https://api.census.gov/data/2024/acs/acs5", the ACS 5-year estimates for 2020-2024
# Your code here

In [ ]:
# groups_response: request census_base + "/groups.json" with headers=HEADERS (the catalog needs no key)
# Print its status code and the first 200 characters of its text
# Your code here

In [ ]:
# catalog: the response's .json(); show its keys
# Your code here

In [ ]:
# groups: a DataFrame of catalog["groups"], keeping its "name" and "description"; print how many; show 3
# Your code here

In [ ]:
# Given, only if the cells above stopped: remove the "#" from the next line, and run this cell.
# It loads the catalog saved on 2026-11-10 by the key's code (1,199 groups, as the live request returns)
# groups = pd.read_csv(checkpoints + "lab11_groups.csv")

In [ ]:
# match: True where groups["description"] contains "median household income" (case=False)
# Show groups[match], sorted by "name"
# Your code here

In [ ]:
# Given: the search as a function. find_groups("internet") shows the groups whose description has those words
# (regex=False reads your words as they are, parentheses included)
def find_groups(words):
    match = groups["description"].str.contains(words, case=False, regex=False)
    return groups[match].sort_values("name")

### ⏱ Microsprint (4 minutes, alone or in pairs)

A table for a question of your own. `find_groups()` is a **function**: a name for the search above, which runs it on whatever words you put between its parentheses.

**Do:** search for a topic of your own, such as "median gross rent", "travel time to work", or "internet". The key searches for "median gross rent".
**Share:** your words, how many groups matched, and the one you'd use.

In [ ]:
# Microsprint: find_groups("your words")
# Your code here

<details>
<summary>💬 Discuss</summary>

- `B19013` came back with nine siblings, `B19013A` to `B19013I`. What do the letters add, and when would you want one of them instead?
- The website says *table* and the API says *group*. What else did you translate from the website's words to the API's?
- The lab saves a copy of this catalog in case your request fails. What can a saved copy not tell you that the live request can?

</details>

## Step 2 — A group's variables (9 minutes)

A group holds **variables**, the columns of the website's table, listed at `groups/B19013.json` ([as a page](https://api.census.gov/data/2024/acs/acs5/groups/B19013.html)). A name is the group, a line and a letter: `B19013_001E` is line 1's **E**stimate, `B19013_001M` its **M**argin of error, the website's "±"; `EA` and `MA` are annotations. Go down the JSON one level a cell, as in ["From the Group to Its Variables"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#from-the-group-to-its-variables). The given `show_estimates()` repeats it for any group: its `for` loop runs the indented lines once for each name.

In [ ]:
# b19013_response: request census_base + "/groups/B19013.json"; b19013: its .json(); show its keys
# Your code here

In [ ]:
# group_vars: b19013["variables"]; show its names with sorted()
# Your code here

In [ ]:
# Show what the catalog says about "B19013_001E"
# Your code here

In [ ]:
# Given: show_estimates(group) asks for a group's catalog and prints each estimate, a name ending in "E", and
# its label. NAME ends in "E" too, but it's the place's name, not an estimate, so it's skipped
def show_estimates(group):
    response = requests.get(census_base + "/groups/" + group + ".json", headers=HEADERS)
    variables = response.json()["variables"]
    for name in sorted(variables):
        if name.endswith("E") and name != "NAME":
            print(name, "|", variables[name]["label"])

show_estimates("B19013")

### ⏱ Microsprint (3 minutes, alone or in pairs)

A table with more lines: group `B25034`, Year Structure Built, which we use on Wednesday. The `!!` in a label separates its levels, the way the website indents a table's rows.

**Do:** run `show_estimates()` on `"B25034"`.
**Share:** how many estimates it has, and which line counts homes built in the 1970s.

In [ ]:
# Microsprint: show_estimates("B25034")
# Your code here

<details>
<summary>💬 Discuss</summary>

- The website shows an estimate and its margin side by side; the API makes them two variables. Why might an API split them?
- `show_estimates()` skips `NAME`, though its name ends in `E`. Without that test, how many lines would B25034 seem to have?
- Which variables would you ask for to rebuild the website's table exactly?

</details>

## Step 3 — The website's number, from the API (11 minutes)

A request names the variables with `get`, the place with `for`, what that place sits inside with `in`, and you with `key` (the API's [user guide](https://www.census.gov/data/developers/guidance/api-user-guide.html)). Places are [FIPS codes](https://www.census.gov/library/reference/code-lists/ansi.html), strings of digits: Colorado is state `08`, and Boulder County is county `013` within it. The answer is a list of lists: row 0 holds the column names, rows 1 onward (`rows[1:]`) the places, and every value is a string. See ["The Same Numbers from the API"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#the-same-numbers-from-the-api).

In [ ]:
# Given: the request's parameters, a dictionary. Its keys are in quotes:
# for and in are words Python keeps for itself
county_params = {
    "get": "NAME,B19013_001E,B19013_001M",   # the place's name, the estimate, and its margin of error
    "for": "county:013",                     # Boulder County...
    "in": "state:08",                        # ...in Colorado
    "key": census_key,
}

In [ ]:
# county_response: requests.get() census_base with params=county_params and HEADERS; print status, text[:200]
# Your code here

In [ ]:
# county_rows: the response's .json(); show it
# Your code here

In [ ]:
# county: a DataFrame of county_rows[1:], with county_rows[0] as its columns; show it
# Your code here

In [ ]:
# Given, only if the cells above stopped: remove the "#" from the next line, and run this cell.
# It loads the row saved on 2026-11-10 by the key's code; dtype=str keeps "08" and "013" as text
# county = pd.read_csv(checkpoints + "lab11_county.csv", dtype=str)

In [ ]:
# Turn the columns "B19013_001E" and "B19013_001M" into numbers with pd.to_numeric; show county
# Your code here

<details>
<summary>💬 Discuss</summary>

- Did the API's number match the website's, \$103,994 ± \$2,268? If it hadn't, what would you check first: the dataset's year, the variable, or the place?
- Why must `08` stay a string? What would `int("08")` lose?
- In week 7, the Wayback Machine's CDX API answered in the same shape: a list of lists whose first row is the header. Why might an API send rows that way rather than as a list of dictionaries?

</details>

## Step 4 — A city, a state, and when the API says no (13 minutes)

The city of Boulder is place `07850`. Ask for it without `in`, and read the answer: a status of [400](https://developer.mozilla.org/en-US/docs/Web/HTTP/Reference/Status/400) comes with the Bureau's reason in words. A place needs its state, because place codes are numbered within each state. The given `look()` holds the two lines you typed; add `in` and look again. Then a given loop asks for a state, a county and a city, as in ["The Same Variable for a State, a County, and a City"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#the-same-variable-for-a-state-a-county-and-a-city).

In [ ]:
# Given: the city of Boulder, without "in"
city_params = {"get": "NAME,B19013_001E,B19013_001M", "for": "place:07850", "key": census_key}

In [ ]:
# city_response: send city_params to census_base; print its status code and text[:200]
# Your code here

In [ ]:
# Given: the two lines you just typed, as a function. look(params) sends params and prints what came back
def look(params):
    response = requests.get(census_base, params=params, headers=HEADERS)
    print(response.status_code, response.text[:200])

In [ ]:
# Add "in": set city_params["in"] to "state:08", then look() at city_params
# Your code here

In [ ]:
# Given: the same two variables for a state, a county, and a city. Only "for" and "in" change.
# .update() adds one dictionary's keys to another; [1][:3] is row 1's first three values;
# time.sleep(1) pauses a second between requests
geographies = [
    {"for": "state:08"},                         # Colorado
    {"for": "county:013", "in": "state:08"},     # Boulder County, in Colorado
    {"for": "place:07850", "in": "state:08"},    # the city of Boulder, in Colorado
]
three = []
for geography in geographies:
    geo_params = {"get": "NAME,B19013_001E,B19013_001M", "key": census_key}
    geo_params.update(geography)
    geo_rows = requests.get(census_base, params=geo_params, headers=HEADERS).json()
    three.append(geo_rows[1][:3])
    time.sleep(1)
pd.DataFrame(three, columns=["NAME", "B19013_001E", "B19013_001M"])

In [ ]:
# Given: try_params, a fresh copy of Step 3's parameters, for the microsprint. Run this again to start over
try_params = {"get": "NAME,B19013_001E,B19013_001M", "for": "county:013", "in": "state:08",
              "key": census_key}

### ⏱ Microsprint (5 minutes, alone or in pairs)

Break it on purpose. Each mistake below makes Step 3's `.json()` stop with the same error, `JSONDecodeError`; what came back before it says which mistake it was.

**Do:** make one mistake in `try_params`, and `look()` at it:
- drop a zero from a code: `try_params["for"] = "county:13"` (the key's choice);
- misspell the variable: `try_params["get"] = "NAME,B19013_01E"`;
- put a space after the comma: `try_params["get"] = "NAME, B19013_001E"`;
- put the key's name in quotes: `try_params["key"] = "census_key"`.

**Share:** your mistake, the status code, and the first words of the text.

In [ ]:
# Microsprint: change one value in try_params (see the list above), then look(try_params)
# Your code here

<details>
<summary>💬 Discuss</summary>

- Across the room, a mistake came back three ways: a 400 with a reason, a [204](https://developer.mozilla.org/en-US/docs/Web/HTTP/Reference/Status/204) with nothing at all, and a 200 with a web page for a bad key. In week 5 we read status codes: why isn't checking for 200 enough here?
- None of our cells handles an error; the chapter's code calls `raise_for_status()`, which stops on a 4xx or 5xx status. Which of today's mistakes would it catch, and which would get past it?
- The city's margin of error is about ten times the state's. Why does a smaller place get a wider margin?

</details>

# Part 2 — Wednesday: many places, and whether to trust a number

One request can ask for several places, or for every place in a state. Then two questions before trusting a number: is a difference bigger than its margin of error, and is the number a number at all?

## Step 5 — Set up again (4 minutes)

Part 2 starts fresh, so a student who missed Monday can start here: nothing from Monday's cells is needed. Put your email in the first given cell, and run both. They import what we need, check and read `CENSUS_KEY` (Step 0 says what to do if it says `MISSING`), and set `census_base`, the address of the ACS 5-year estimates for 2020–2024.

In [ ]:
# Given: imports; your User-Agent (put your email in it); a check that CENSUS_KEY is set, not the key
import os
import requests
import pandas as pd
import matplotlib.pyplot as plt

HEADERS = {"User-Agent": "WebDataScience/1.0 (your-email@colorado.edu)"}
checkpoints = "https://raw.githubusercontent.com/cuinfoscience/INFO4617-Fall2026/main/labs/week-11/data/"
census_base = "https://api.census.gov/data/2024/acs/acs5"
print("CENSUS_KEY:", "set" if os.environ.get("CENSUS_KEY") else "MISSING")

In [ ]:
# Given: ask for the key with getpass only if it's MISSING (Step 0), then read it
from getpass import getpass
if not os.environ.get("CENSUS_KEY"):
    os.environ["CENSUS_KEY"] = getpass("CENSUS_KEY isn't set. Paste it here: ")
census_key = os.environ["CENSUS_KEY"]

<details>
<summary>💬 Discuss</summary>

- Did your key work today? If it didn't on Monday, which cause in the chapter's "When a Key Says `MISSING`" was it?

</details>

## Step 6 — Several places in one request (16 minutes)

Put several codes in `for`, separated by commas, and one request returns a row for each: `place:07850,45970` is Boulder and Longmont, as in ["Comparing Places"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#comparing-places). Monday's cells are gathered in a given function, `census_get()`: it sends the parameters, prints the status, makes the DataFrame, and turns each variable's column (each name starting with `B`) into numbers. `def` names the lines, and `return` hands back the result. The given chart, the chapter's [figure 11.4](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#fig-housing-boulder-longmont), uses it for ten variables at once.

In [ ]:
# Given: Monday's by-hand cells, as one function. census_get(params) returns the answer as a DataFrame
def census_get(params):
    response = requests.get(census_base, params=params, headers=HEADERS)
    print("status", response.status_code)
    rows = response.json()
    table = pd.DataFrame(rows[1:], columns=rows[0])
    for column in table.columns:
        if column.startswith("B"):
            table[column] = pd.to_numeric(table[column])
    return table

pair_params = {"get": "NAME,B19013_001E,B19013_001M", "for": "place:07850", "in": "state:08",
               "key": census_key}

In [ ]:
# Set pair_params["for"] to Boulder and Longmont: "place:07850,45970"
# Your code here

In [ ]:
# incomes: census_get(pair_params); show it
# Your code here

In [ ]:
# Given: the chapter's figure 11.4. housing_vars, a list comprehension, writes the ten codes
# B25034_002E to _011E; labels keeps each label's last part, after its last "!!";
# .T turns the table, and iloc[::-1] puts the oldest decade first
housing_vars = [f"B25034_{line:03d}E" for line in range(2, 12)]
housing_params = {"get": "NAME," + ",".join(housing_vars), "for": "place:07850,45970", "in": "state:08",
                  "key": census_key}
df_compare = census_get(housing_params)
b25034 = requests.get(census_base + "/groups/B25034.json", headers=HEADERS).json()["variables"]
labels = {var: b25034[var]["label"].split("!!")[-1] for var in housing_vars}
housing = df_compare.set_index("NAME")[housing_vars].rename(columns=labels).T

fig, ax = plt.subplots(figsize=(8, 4.5))
housing.iloc[::-1].plot.bar(ax=ax)
ax.set(title="Housing units by year built: Boulder and Longmont, ACS 2020-2024",
       ylabel="Housing units", xlabel="")
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
ax.legend(title="")
plt.show()

In [ ]:
# Given: for five places, all housing units (B25034_001E) and those built since 2000 (_002E to _004E):
# Boulder, Longmont, Fort Collins (27425), Lakewood (43000), and Castle Rock (12415)
five_params = {"get": "NAME,B25034_001E,B25034_002E,B25034_003E,B25034_004E",
               "for": "place:07850,45970,27425,43000,12415", "in": "state:08", "key": census_key}
five = census_get(five_params)
five["since_2000"] = five["B25034_002E"] + five["B25034_003E"] + five["B25034_004E"]
five

### ⏱ Microsprint (5 minutes, alone or in pairs)

A short version of Additional Exercise 1. Cities differ in size, so counts don't compare fairly; shares do.

**Do:** divide each place's units built since 2000 by its total, `B25034_001E`.
**Share:** the place with the largest share, the one with the smallest, and the two shares.

In [ ]:
# Microsprint: five["share"]: five["since_2000"] divided by five["B25034_001E"]; show NAME and share
# Your code here

<details>
<summary>💬 Discuss</summary>

- Longmont has more units than Boulder from every decade since the 1990s. The chapter names one likely reason, Boulder's 1976 cap on building permits. What would you need to know to say it was the reason?
- The chart counts the units standing in 2020–2024 by the year they were built. What does it leave out?
- `census_get()` has no error handling. If the API answered with a 429 or a maintenance page, which of its lines would stop, and what would the error say?

</details>

## Step 7 — Is the difference real? (10 minutes)

Longmont's median household income is higher than Boulder's, but each is an estimate with a margin of error, at 90% confidence. Don't judge by whether the two ranges overlap: the ACS handbook says that test isn't reliable ([chapter 7, p. 55](https://www.census.gov/content/dam/Census/library/publications/2020/acs/acs_general_handbook_2020_ch07.pdf)). Compare the difference with its own margin: square the two margins (`** 2`), add them, and take the square root (`** 0.5`). A difference smaller than its margin may be no difference at all.

In [ ]:
# estimates: incomes["B19013_001E"]; difference: Longmont's (row 1) minus Boulder's (row 0); print it
# Your code here

In [ ]:
# margins: incomes["B19013_001M"]; margin_of_difference: (margins[0] ** 2 + margins[1] ** 2) ** 0.5; print it
# Your code here

<details>
<summary>💬 Discuss</summary>

- The difference is smaller than its margin. What can you say about which city's households earn more, and what can't you?
- Boulder's margin is about twice Longmont's. What about the city could make its estimate less certain?
- Which checks like this one could a pipeline run on its own, every time it runs?

</details>

## Step 8 — Every place, and numbers that aren't (20 minutes)

A wildcard, `place:*` with `in` set to `state:08`, asks for every place in Colorado: its cities, towns, and census-designated places. Ask for median gross rent, group `B25064`, and [`describe()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.describe.html) the column before you trust it. Where too few households answered, the API sends a code, `-666666666`, not a rent; the Bureau's [notes](https://www.census.gov/data/developers/data-sets/acs-1year/notes-on-acs-estimate-and-annotation-values.html) list each code. [`mask()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.mask.html) turns those rows into `NaN`, a missing value, as in ["Every Place in a State, and a Number That Isn't"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#every-place-in-a-state-and-a-number-that-isnt).

In [ ]:
# Given: median gross rent (B25064_001E) and its margin of error, for the city of Boulder
rent_params = {"get": "NAME,B25064_001E,B25064_001M", "for": "place:07850", "in": "state:08",
               "key": census_key}

In [ ]:
# Set rent_params["for"] to every place, "place:*"; places: census_get(rent_params); print how many rows
# Your code here

In [ ]:
# Given, only if the cell above stopped: remove the "#" from the next line, and run this cell.
# It loads the 482 places saved on 2026-11-10 by the key's code, with the place codes kept as text
# places = pd.read_csv(checkpoints + "lab11_places.csv", dtype={"state": str, "place": str})

In [ ]:
# describe() the column "B25064_001E", rounded with .round(): look at its min and its max
# Your code here

In [ ]:
# no_estimate: True where "B25064_001E" equals -666666666; print how many places have no estimate
# Your code here

In [ ]:
# places["rent"]: "B25064_001E" with the no_estimate rows masked; describe() it, rounded
# Your code here

In [ ]:
# top: places sorted by "rent" (ascending=False), its first 10 rows; show NAME, rent, B25064_001M
# Your code here

In [ ]:
# Given: the ten as horizontal bars, each with a line as wide as its margin of error on either side.
# A margin below zero is a code too, so it's masked; invert_yaxis() puts the highest at the top
top_margin = top["B25064_001M"].mask(top["B25064_001M"] < 0)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(top["NAME"], top["rent"], xerr=top_margin, color="tab:blue", ecolor="black", capsize=3)
ax.invert_yaxis()
ax.set(title="Highest median gross rent, Colorado places, ACS 2020-2024",
       xlabel="Median gross rent (dollars a month), with its margin of error", ylabel="")
plt.show()

### ⏱ Microsprint (4 minutes, alone or in pairs)

The margin column has a code of its own.

**Do:** find the places whose margin, `B25064_001M`, is `-333333333`, and look the code up in the Bureau's [notes](https://www.census.gov/data/developers/data-sets/acs-1year/notes-on-acs-estimate-and-annotation-values.html).
**Share:** how many places, their median rent, and what the code says about it.

In [ ]:
# Microsprint: capped: True where "B25064_001M" equals -333333333; show places[capped]
# Your code here

<details>
<summary>💬 Discuss</summary>

- More than a third of Colorado's places have no estimate of median gross rent. Left in, what would `-666666666` have done to a mean, and to a ranking?
- The chart's top three are capped at 3,501, "\$3,500 or more", and El Jebel's margin is ±\$2,790. Which of the ten could you rank with confidence?
- Typed without `in`, `county:*` returns every county in the country (3,222 when this lab was written), with no error. How would you notice before your chart did?

</details>

# Part 3 — Friday: FRED, and let the server compute

FRED (Federal Reserve Economic Data), from the Federal Reserve Bank of St. Louis, holds about 800,000 economic series, and it can transform one before it sends it: a change, a percent change, a lower frequency. Asking the server, instead of downloading everything and doing it in pandas, is today's idea, and so is checking it once.

## Step 9 — Set up again (3 minutes)

Part 3 starts fresh and needs nothing from Monday or Wednesday. Put your email in the first given cell, and run both: they import what we need and check `FRED_KEY` (Step 0 says what to do if it says `MISSING`). Then read the key.

In [ ]:
# Given: imports; your User-Agent (put your email in it); a check that FRED_KEY is set, not the key
import os
import requests
import pandas as pd
import matplotlib.pyplot as plt

HEADERS = {"User-Agent": "WebDataScience/1.0 (your-email@colorado.edu)"}
checkpoints = "https://raw.githubusercontent.com/cuinfoscience/INFO4617-Fall2026/main/labs/week-11/data/"
print("FRED_KEY:", "set" if os.environ.get("FRED_KEY") else "MISSING")

In [ ]:
# Given: only if the check said MISSING, ask for the key with getpass (Step 0). Otherwise it does nothing
from getpass import getpass
if not os.environ.get("FRED_KEY"):
    os.environ["FRED_KEY"] = getpass("FRED_KEY isn't set. Paste it here: ")

In [ ]:
# fred_key: read "FRED_KEY" from os.environ with square brackets (never print it)
# Your code here

<details>
<summary>💬 Discuss</summary>

- The Census Bureau's key went in a parameter named `key`; FRED's is named `api_key`. What should you read before you send a key to a new API?

</details>

## Step 10 — A series, and what it is (11 minutes)

FRED names each series with an ID, shown on its web page ([figure 11.2](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#fig-fred-series)): Boulder County's median household income is `MHICO08013A052NCEN`. First ask the [`series` endpoint](https://fred.stlouisfed.org/docs/api/fred/series.html) what it is. FRED reports a mistake as JSON with an `error_message`, so print that before reading on. Then ask [`series/observations`](https://fred.stlouisfed.org/docs/api/fred/series_observations.html) for the values. FRED sends each value as a string, and a missing one as `"."`, which [`pd.to_numeric(errors="coerce")`](https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html) turns into `NaN`. See ["Finding Series IDs"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#finding-series-ids).

In [ ]:
# fred_base: "https://api.stlouisfed.org/fred"
# Your code here

In [ ]:
# Given: which series, your key, and JSON (without "file_type", FRED answers in XML)
series_params = {"series_id": "MHICO08013A052NCEN", "api_key": fred_key, "file_type": "json"}

In [ ]:
# series_url: fred_base + "/series"; series_response: send series_params to it; series_json: its .json()
# Print the status code and series_json.get("error_message"), which is None when there's no error
# Your code here

In [ ]:
# about: series_json["seriess"][0], the one series; print its "title", "frequency" and "units"
# Your code here

In [ ]:
# Given: the same series, up to a fixed end, so a rerun gets the same answer
income_params = {"series_id": "MHICO08013A052NCEN", "api_key": fred_key, "file_type": "json",
                 "observation_end": "2024-12-31"}

In [ ]:
# obs_url: fred_base + "/series/observations"; income_response: send income_params to it
# Your code here

In [ ]:
# income_json: the response's .json(); income: a DataFrame of income_json["observations"]
# Keep its "date" and "value" columns; show the first 8 rows
# Your code here

In [ ]:
# Convert "value" with pd.to_numeric(..., errors="coerce") and "date" with pd.to_datetime; show the last 3 rows
# Your code here

<details>
<summary>💬 Discuss</summary>

- The Census API answered with a list of lists; FRED with a dictionary whose `observations` is a list of dictionaries. Which turned into a DataFrame more easily, and why?
- FRED's 2024 figure for Boulder County is \$103,293; Monday's ACS figure for 2020–2024 was \$103,994. The chapter says they come from different programs. Which would you cite, and how would you say which it is?
- Why fix `observation_end` at 2024-12-31, when the series ends in 2024 anyway?

</details>

## Step 11 — Let the server compute: `units` (6 minutes)

We'll ask for observations several more times, changing one parameter each time, so the given `fred_observations()` gathers Step 10's lines, as the chapter's does. `**params` collects any other parameters you name, such as `units="chg"`, and sends them along. [`units`](https://fred.stlouisfed.org/docs/api/fred/series_observations.html#units) asks FRED to transform the values first: `chg` is the change from the previous observation, and `pch` the percent change. See ["Let the Server Compute: `units`"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#let-the-server-compute-units).

In [ ]:
# Given: Step 10's lines, as the chapter's function. Any parameter you name after the ID goes into params
def fred_observations(series_id, **params):
    params.update(series_id=series_id, api_key=fred_key, file_type="json")
    response = requests.get(fred_base + "/series/observations", params=params, headers=HEADERS)
    df = pd.DataFrame(response.json()["observations"])[["date", "value"]]
    df["date"] = pd.to_datetime(df["date"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    return df

In [ ]:
# change: fred_observations() for "MHICO08013A052NCEN" with units="chg" and observation_end="2024-12-31"
# Your code here

In [ ]:
# Given: the percent change too, and the three answers side by side, one row a year
growth = fred_observations("MHICO08013A052NCEN", units="pch", observation_end="2024-12-31")
views = pd.DataFrame({"year": income["date"].dt.year, "dollars": income["value"],
                      "change": change["value"], "percent change": growth["value"]})
views.head(10).round(1)

### ⏱ Microsprint (3 minutes, alone or in pairs)

Read what the server computed.

**Do:** find the years in which Boulder County's median household income *fell*: the rows of `views` whose `change` is below zero.
**Share:** how many years, which ones, and the largest fall in dollars.

In [ ]:
# Microsprint: fell: True where views["change"] is below zero; show views[fell]
# Your code here

<details>
<summary>💬 Discuss</summary>

- `change` is `NaN` for 1993, 1995 and 1997, though they have dollars. Why, and could income have fallen in a year our list of falls doesn't show?
- FRED computed `change` from the same dollars we have. When is it worth asking the server for something you could compute yourself?
- If FRED answered with an error, which line of `fred_observations()` would stop, and would its error say what FRED said?

</details>

## Step 12 — Let the server compute: `frequency` (9 minutes)

A series' **frequency** is how often it's measured, and FRED can lower it before sending: [`frequency="a"`](https://fred.stlouisfed.org/docs/api/fred/series_observations.html#frequency) with `aggregation_method="avg"` averages each year's months. Take `CPIAUCNS`, the national Consumer Price Index (CPI), which is monthly, over a fixed window set by `observation_start` and `observation_end`. Look for a missing month before you trust a yearly average. See ["Let the Server Compute: `frequency` and `aggregation_method`"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#let-the-server-compute-frequency-and-aggregation_method).

In [ ]:
# start: "2015-01-01"; end: "2025-12-31"
# Your code here

In [ ]:
# monthly: fred_observations() for "CPIAUCNS" with observation_start=start and observation_end=end; print len()
# Your code here

In [ ]:
# Given, only if the cell above stopped: remove the "#" from the next line, and run this cell.
# It loads the 132 months saved on 2026-11-10 by the key's code, already converted
# monthly = pd.read_csv(checkpoints + "lab11_cpi_monthly.csv", parse_dates=["date"])

In [ ]:
# missing: True where monthly["value"] is NaN (.isna()); show monthly[missing]
# Your code here

In [ ]:
# annual: the same, with frequency="a" and aggregation_method="avg" as well; show its last 3 rows
# Your code here

In [ ]:
# Given: with units="pc1" too, the percent change from the year before: each year's inflation. FRED applies
# the frequency first, then the units
inflation = fred_observations("CPIAUCNS", frequency="a", aggregation_method="avg", units="pc1",
                              observation_start=start, observation_end=end)
table = pd.DataFrame({"year": annual["date"].dt.year, "mean index": annual["value"],
                      "inflation (%)": inflation["value"]})
table.round(1)

In [ ]:
# Given: the monthly series over time. A NaN leaves a gap in the line
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(monthly["date"], monthly["value"])
ax.set(title="Consumer Price Index for all urban consumers, U.S., monthly (CPIAUCNS)",
       ylabel="Index, 1982-84 = 100", xlabel="")
plt.show()

### ⏱ Microsprint (3 minutes, alone or in pairs)

Trust the server, but check it once.

**Do:** from `monthly`, compute each year's mean and count of values with [`groupby()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html) on the year, `monthly["date"].dt.year`, and compare the means with `table`.
**Share:** your 2024 and 2025 means beside FRED's, and how many months each is the mean of.

In [ ]:
# Microsprint: by_year: monthly grouped by monthly["date"].dt.year, its "value" column
# Print the last 3 rows of by_year.mean(), rounded to 1 place, and of by_year.count()
# Your code here

<details>
<summary>💬 Discuss</summary>

- FRED's 2025 mean and ours agree. How many months is each the mean of? Would you call 2.6 percent "2025's inflation" without a footnote?
- `aggregation_method` could also be `sum` or `eop`, the end of the period. Which suits a price index, and which a count, like jobs added?
- Misspelled as `observation_begin`, `observation_start` is ignored without a word: FRED sends every month since 1913 (1,356 when this lab was written). How would you notice?

</details>

## Step 13 — Denver against the nation (3 minutes)

With one series understood, comparing two is asking each the same question. The Bureau of Labor Statistics publishes a CPI for the Denver area, which FRED carries as `CUUSA433SA0`. It's annual already, so it needs only `units="pc1"`. The given cell draws the chapter's [figure 11.5](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#fig-cpi-denver-national): one pair of bars a year, with a dashed line at 2 percent, the Federal Reserve's target.

In [ ]:
# Given: Denver's inflation, merged with the nation's on the date, as paired bars
denver = fred_observations("CUUSA433SA0", units="pc1", observation_start=start, observation_end=end)
cpi = denver.merge(inflation, on="date", suffixes=("_denver", "_national"))
pairs = cpi.set_index(cpi["date"].dt.year)[["value_denver", "value_national"]]
pairs.columns = ["Denver-Aurora-Lakewood", "United States"]

fig, ax = plt.subplots(figsize=(8, 4))
pairs.plot.bar(ax=ax, rot=0, width=0.8)
ax.axhline(y=2, color="gray", linestyle="--", linewidth=1, label="2% target")
ax.set(title="Annual inflation (CPI-U, percent change from the year before)", ylabel="Percent", xlabel="")
ax.legend()
plt.show()

<details>
<summary>💬 Discuss</summary>

- In which years did Denver's prices rise faster than the nation's, and in which slower?
- Denver's series is annual and the nation's monthly. What would have gone wrong had we merged the monthly series, without `frequency="a"`?

</details>

## Step 14 — What did we learn? (3 minutes)

In a few sentences, as comments:

- Step 3 asked the API for the website's number. Did they match, and what would you check first if they hadn't?
- On Monday, a mistake came back three ways. Which is hardest to notice, and what habit catches it?
- In Step 8, how many of Colorado's places had no estimate of median gross rent, and what did `describe()` show before you masked them?
- Which of this week's numbers would be different if you ran this notebook next week, and which wouldn't? Why?

**Recommended Exercises ahead** (in the chapter's companion notebook, due Sunday):

- Step 1, load your keys without printing them: Step 0's check and `os.environ[...]` (the exercise asks you to print each key's length; our check printed `set`).
- Step 2, one ACS variable for every Colorado county: Steps 1–2 to find the group and the variable, Step 3's `for` and `in`, and Step 8's wildcard, which asked for every place.
- Step 3, build the DataFrame, convert, sort, and print the extremes: Step 3's DataFrame and `pd.to_numeric`, and Step 8's `describe()`, codes, `mask()` and `sort_values()`.
- Step 4, a horizontal bar chart of the top 20 counties: Step 8's sort, `head()` and `barh()` chart.
- Step 5, a related FRED series for Colorado over time: Steps 10–12's `fred_observations()`, `observation_start`, and the line plot of `monthly`; find the series on FRED's website, as Step 10 did.
- Step 6, interpret: the Discuss questions after Steps 3, 7, 8 and 12.

**Further, on your own** (not graded): the chapter's third API, the U.S. Energy Information Administration's, in ["EIA API v2"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#eia-api-v2): routes that describe themselves, filters applied on the server, and paging through a large answer. ["Combining Census and EIA Data"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#combining-census-and-eia-data) then joins Census incomes with the EIA's electricity bills, state by state, through a crosswalk. Its key, `EIA_KEY`, comes from the [EIA](https://www.eia.gov/opendata/register.php).

In [ ]:
# Your answers here, as comments


## Friday: your portfolio checkpoint (15 minutes)

The last 15 minutes go to your project. This week's checkpoint, **Set up your project repo**, is due Sunday, November 1:

- **a public repository** on your own GitHub account, with a descriptive name;
- **`README.md`**, with a title, your name, a one-paragraph summary of your research question, and placeholder sections: Data, Methods, Findings, How to reproduce;
- **`proposal.md`**: your proposal, revised if you've had feedback;
- **placeholders for the rest**: a notebook (`.ipynb`), a `report.md` (or a note that your report will be a PDF), and a `data/` folder.

Use the time to make the repository and push its first commit, here, where you can ask. If your project uses an API key, this week's rule holds there too: the key stays in an environment variable, never in the repository.